# Bright TV User Profile & Viewership Analysis — Pandas Notebook

This notebook converts the supplied SQL workflow into **Python/Pandas**, with the analysis broken into logical cells.  
Each code cell contains comments explaining what the code does.

> **Assumption:** The source data is available as CSV or Excel files. Update the file paths in the import cell if required.

In [0]:
# Import the main Python libraries used for data analysis.
import pandas as pd
import numpy as np


## Calling user_profile table

In [0]:
user_profile_df = spark.table("bright.bright_tv.bright_tv_dataset_user_profiles")
user_profile = user_profile_df.toPandas()
display(user_profile)


## Calling viewership table

In [0]:
viewership_df = spark.table("bright.bright_tv.bright_tv_dataset_viewership")
viewership = viewership_df.toPandas() #converted to pandas- so you can use padas functions.
display(viewership)

# checking for duplicates

In [0]:
# Check for duplicate UserIDs.
# This is the Pandas equivalent of GROUP BY UserID HAVING COUNT(*) > 1.
# 1. Count total duplicate rows
print(user_profile.duplicated().sum())

# 2. Extract and view the actual duplicate rows
print(user_profile[user_profile.duplicated()])


# checking for duplicates

In [0]:
# Check for duplicate UserIDs.
# This is the Pandas equivalent of GROUP BY UserID HAVING COUNT(*) > 1.
# 1. Count total duplicate rows
print(viewership.duplicated().sum())

# 2. Extract and view the actual duplicate rows
display(viewership[viewership.duplicated()])


In [0]:
viewership = viewership.drop_duplicates()

# Dropping duplicates- notice how the total number of rows changed

In [0]:
display(viewership)

#user_profile analysis

# measuring data size

In [0]:
# Measure the size of the user profile dataset.
# number_of_rows = total records.
# number_subs = number of unique subscribers/UserIDs.
number_of_rows = len(user_profile)
number_subs = user_profile["UserID"].nunique(dropna=True)

print("Number of rows:", number_of_rows)
print("Number of unique subscribers:", number_subs)


# checking for null values

In [0]:
# Count rows where UserID is missing/null.
# This is the Pandas equivalent of WHERE UserID IS NULL.
null_userid_count = user_profile["UserID"].isna().sum()

print("Rows with NULL UserID:", null_userid_count)


## Gender Analysis

In [0]:
# -------------------------------
# GENDER CHECKS
# -------------------------------

# Show all distinct Gender values before cleaning.
display(user_profile["Gender"].drop_duplicates().to_frame(name="Gender"))


In [0]:
# Count missing/null Gender values.
gender_null_count = user_profile["Gender"].isna().sum()

print("Rows where Gender is NULL:", gender_null_count)


In [0]:
# Count records where Gender contains a blank/whitespace value.
# strip() removes spaces so values such as "   " are treated as blank.
gender_blank_count = (
    user_profile["Gender"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("Rows where Gender is blank:", gender_blank_count)


In [0]:
# Standardise Gender values.
# Blank values and the text "None" are converted to "Unknown".
user_profile["Gender_Clean"] = (
    user_profile["Gender"]
    .fillna("")
    .astype(str)
    .str.strip()
    .replace({"": "Unknown", "None": "Unknown"})
)

# Count unique subscribers by cleaned gender.
gender_summary = (
    user_profile.groupby("Gender_Clean")["UserID"]
    .nunique()
    .reset_index(name="Subs")
    .sort_values("Subs", ascending=False)
)

display(gender_summary)


##race analysis

In [0]:
# -------------------------------
# RACE CHECKS
# -------------------------------

# Display the distinct Race values before cleaning.
display(user_profile["Race"].drop_duplicates().to_frame(name="Race"))


In [0]:
# Count rows where Race is NULL.
race_null_count = user_profile["Race"].isna().sum()

print("Rows where Race is NULL:", race_null_count)


In [0]:
# Clean Race values.
# "other", blank values and "None" are standardised to "None".
user_profile["Race"] = (
    user_profile["Race_Clean"]
    .fillna("")
    .astype(str)
    .str.strip()
    .replace({
        "other": "None",
        "Other": "None",
        "": "None"
    })
)

# Display the cleaned distinct Race values.
display(user_profile["Race_Clean"].drop_duplicates().to_frame(name="Race"))


In [0]:
display(user_profile)

# province/ region analysis

In [0]:
# -------------------------------
# PROVINCE / REGION CHECKS
# -------------------------------

# Display the distinct Province values before cleaning.
display(user_profile["Province"].drop_duplicates().to_frame(name="Province"))


In [0]:
# Clean Province into a Region field.
# Blank values and "None" are classified as "Uncategorized".
user_profile["Region"] = (
    user_profile["Province"]
    .fillna("")
    .astype(str)
    .str.strip()
    .replace({
        "": "Uncategorized",
        "None": "Uncategorized"
    })
)

# Display the cleaned regions.
display(user_profile["Region"].drop_duplicates().to_frame(name="Region"))


# age analysis

In [0]:
# -------------------------------
# AGE CHECKS
# -------------------------------

# Convert Age to numeric so that minimum, maximum and missing-value checks work correctly.
user_profile["Age"] = pd.to_numeric(user_profile["Age"], errors="coerce")

# Find the minimum and maximum age.
min_age = user_profile["Age"].min()
max_age = user_profile["Age"].max()

print("Minimum age:", min_age)
print("Maximum age:", max_age)


In [0]:
# Count records where Age is NULL/missing.
age_null_count = user_profile["Age"].isna().sum()

print("Rows where Age is NULL:", age_null_count)


In [0]:
# Create age groups using the same business rules as the SQL CASE statement.
def create_age_group(age):
    if pd.isna(age):
        return "Unknown"
    elif age == 0:
        return "Infants"
    elif 1 <= age <= 12:
        return "Kids"
    elif 13 <= age <= 19:
        return "Teenager"
    elif 20 <= age <= 35:
        return "Youth"
    elif 36 <= age <= 50:
        return "Adult"
    elif 51 <= age <= 65:
        return "Senior"
    elif age > 65:
        return "Madala"
    else:
        return "Unknown"

user_profile["age_groups"] = user_profile["Age"].apply(create_age_group)

# Display the number of subscribers in each age group.
age_group_summary = (
    user_profile.groupby("age_groups")["UserID"]
    .nunique()
    .reset_index(name="Subscribers")
    .sort_values("Subscribers", ascending=False)
)

display(age_group_summary)


In [0]:
display(user_profile)

# EMAIL AND SOCIAL MEDIA FLAGS

In [0]:
# -------------------------------
# EMAIL AND SOCIAL MEDIA FLAGS
# -------------------------------

# Clean the Email column so blank and "None" values can be identified consistently.
email_clean = (
    user_profile["Email"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Flag a record as 1 when a valid email value exists; otherwise flag it as 0.
user_profile["email_flag"] = (
    email_clean.ne("") & email_clean.ne("None")
).astype(int)

# Clean the Social Media Handle column.
social_clean = (
    user_profile["Social Media Handle"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Flag a record as 1 when a social media handle exists; otherwise flag it as 0.
user_profile["SM_flag"] = (
    social_clean.ne("") & social_clean.ne("None")
).astype(int)

# Review the resulting flags.
display(
    user_profile[["UserID", "email_flag", "SM_flag"]].head(20)
)


#user_profile cte

In [0]:
# Create a clean user profile table.
# This table represents the SQL user_profiles CTE.
user_profile_cte = user_profile[
    [
        "UserID",
        "Region",
        "Age",
        "age_groups",
        "email_flag",
        "SM_flag",
        "Race_Clean",
        "Gender_Clean"
    ]
].copy()

# Rename columns to match the SQL output naming convention.
user_profile_clean = user_profile_cte.rename(columns={
    "Race_Clean": "Race",
    "Gender_Clean": "Gender"
})

display(user_profile_clean.head())


In [0]:
display(user_profile_clean)

# viewership analysis

In [0]:
viewership.head(5)

#record date and duration

In [0]:
# Convert RecordDate2 to a proper Pandas datetime field.
# Invalid dates are converted to NaT so they can be identified and handled.
viewership["RecordDate2"] = pd.to_datetime(
    viewership["RecordDate2"],
    errors="coerce"
)

# Convert Duration 2 to a timedelta.
# Duration 2 is stored as a timestamp with an epoch date (1899-12-31).
# Extract the time component as a timedelta by subtracting the epoch.
epoch = pd.Timestamp('1899-12-31')
viewership["Duration 2"] = pd.to_datetime(
    viewership["Duration 2"],
    errors="coerce"
) - epoch

# Display the converted columns.
display(viewership[["RecordDate2", "Duration 2"]].head())


In [0]:
# Create the subscriber ID used by the viewership data.
# The SQL uses COALESCE(UserID0, userid4, 0), so we take UserID0 when available,
# otherwise userid4, and use 0 when both are missing.
viewership["userid"] = (
    viewership["UserID0"]
    .combine_first(viewership["userid4"])
    .fillna(0)
)

display(viewership[["UserID0", "userid4", "userid"]].head())


In [0]:
viewership.head(5)

#date analysis

In [0]:
# -------------------------------
# VIEWERSHIP DATE FEATURES
# -------------------------------

# Extract the month ID in YYYYMM format.
viewership["month_id"] = viewership["RecordDate2"].dt.strftime("%Y%m")

# Extract the calendar date from the timestamp.
viewership["watch_date"] = viewership["RecordDate2"].dt.date

# Extract the day name, e.g. Monday, Tuesday, etc.
viewership["day_name"] = viewership["RecordDate2"].dt.day_name()

# Extract the day number within the month.
# This mirrors the supplied SQL expression TO_CHAR(RecordDate2, 'DD').
viewership["day_of_week"] = viewership["RecordDate2"].dt.strftime("%d")

# Extract the hour of day.
viewership["hour_of_day"] = viewership["RecordDate2"].dt.hour

# Extract the month name.
viewership["month_name"] = viewership["RecordDate2"].dt.month_name()

display(
    viewership[
        ["RecordDate2", "month_id", "watch_date", "day_name",
         "day_of_week", "hour_of_day", "month_name"]
    ].head()
)


#watch_time buckets

In [0]:
# Create a readable watch-time field in HH:MM:SS format.
viewership["watch_time"] = viewership["RecordDate2"].dt.strftime("%H:%M:%S")

# Classify each viewing record into a time-of-day period.
def classify_time_of_day(hour):
    if pd.isna(hour):
        return "Unknown"
    elif 0 <= hour <= 5:
        return "01. Midnight"
    elif 6 <= hour <= 11:
        return "02. Morning"
    elif 12 <= hour <= 16:
        return "03. Afternoon"
    elif 17 <= hour <= 23:
        return "04. Evening"
    return "Unknown"

viewership["time_of_day"] = viewership["hour_of_day"].apply(
    classify_time_of_day
)

display(
    viewership[
        ["watch_time", "hour_of_day", "time_of_day"]
    ].head(20)
)


#duration buckets

In [0]:
# Calculate viewing duration in minutes.
# Total seconds are divided by 60 and rounded to two decimal places.
viewership["duration_minute"] = (
    viewership["Duration 2"].dt.total_seconds() / 60
).round(2)

# Create a readable duration field in HH:MM:SS format.
viewership["duration"] = viewership["Duration 2"].apply(
    lambda x: str(x).split(" days ")[-1] if pd.notna(x) else None
)

display(
    viewership[
        ["Duration 2", "duration", "duration_minute"]
    ].head(20)
)


#screen time buckets

In [0]:
# Create screen-time buckets based on duration in minutes.
# The original SQL contains an overlapping condition for >60 minutes.
# Here we implement the intended business logic:
# <30 minutes = Low Usage, 30–<60 = Medium Usage, >=60 = High Usage.
def screen_time_bucket(minutes):
    if pd.isna(minutes) or minutes <= 0:
        return "04. No Usage"
    elif minutes < 30:
        return "01. Low Usage <30 min"
    elif minutes < 60:
        return "02. Med Usage <60 min"
    else:
        return "03. High Usage >=60 min"

viewership["screen_time_buckets"] = viewership[
    "duration_minute"
].apply(screen_time_bucket)

display(
    viewership[
        ["duration_minute", "screen_time_buckets"]
    ].head(20)
)


#day classification

In [0]:
# Classify each viewing record as a weekday or weekend.
viewership["day_classification"] = np.where(
    viewership["day_name"].isin(["Saturday", "Sunday"]),
    "weekend",
    "weekday"
)

display(
    viewership[
        ["watch_date", "day_name", "day_classification"]
    ].head(20)
)


#channel analysis

In [0]:
# Standardise TV channel names.
# This combines spelling/casing variants into consistent channel categories.
viewership["Tv_channel"] = (
    viewership["Channel2"]
    .astype("string")
    .str.strip()
)

viewership["Tv_channel"] = viewership["Tv_channel"].replace({
    "Sawsee": "Sawsee",
    "SawSee": "Sawsee",
    "SuperSport Live Events": "Live Events",
    "Live on SuperSport": "Live Events",
    "Supersport Live Events": "Live Events",
    "DStv Events 1": "Live Events"
})

display(
    viewership[["Channel2", "Tv_channel"]]
    .drop_duplicates()
    .sort_values(["Tv_channel", "Channel2"])
)


#viewership cte

In [0]:
# Create the clean viewership table.
# This represents the SQL viewership CTE in Pandas.
viewership_clean = viewership[
    [
        "userid",
        "month_id",
        "watch_date",
        "day_of_week",
        "day_name",
        "day_classification",
        "month_name",
        "Tv_channel",
        "screen_time_buckets",
        "watch_time",
        "hour_of_day",
        "duration",
        "duration_minute",
        "time_of_day"
    ]
].copy()

display(viewership_clean.head())


#left join two tables

In [0]:
# -------------------------------
# JOIN VIEWERSHIP TO USER PROFILE
# -------------------------------

# Make sure both join keys have a consistent type.
viewership_clean["userid"] = viewership_clean["userid"].astype(str)
user_profile_clean["UserID"] = user_profile_clean["UserID"].astype(str)

# Left join viewership to the cleaned user profile.
# This is equivalent to:
# FROM viewership AS A
# LEFT JOIN user_profiles AS B ON A.userid = B.userid
final_data = viewership_clean.merge(
    user_profile_clean,
    how="left",
    left_on="userid",
    right_on="UserID"
)

# Remove the duplicate UserID column after the join.
final_data = final_data.drop(columns=["UserID"])

# Rename userid to sub_id to match the SQL output.
final_data = final_data.rename(columns={"userid": "sub_id"})

display(final_data.head())


In [0]:
# Check how many viewership records successfully matched to a user profile.
# A missing Region/Gender/etc. after the left join may indicate that a subscriber
# exists in the viewing data but not in the user profile table.
matched_profiles = final_data["Region"].notna().sum()
unmatched_profiles = final_data["Region"].isna().sum()

print("Viewership records:", len(final_data))
print("Matched to user profile:", matched_profiles)
print("Unmatched to user profile:", unmatched_profiles)


In [0]:
# Remove exact duplicate rows from the final analytical dataset.
# This is the Pandas equivalent of the intention behind GROUP BY ALL
# in the supplied SQL query.
final_data = final_data.drop_duplicates().reset_index(drop=True)

print("Final analytical dataset shape:", final_data.shape)
display(final_data)


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.